In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# Inputs
# ------------------------------------------------------------
ROUNDS_PATH = Path("/Users/joshmacbook/python_projects/OAD/Data/in Use/combined_rounds_all_2017_2026.csv")
ODDS_XLSX   = Path("/Users/joshmacbook/python_projects/OAD/Data/in Use/Odds_and_Results.xlsx")

CUTOFF_DATE = pd.Timestamp("2026-01-22")  # "before 1/22/2026" => strictly <
EVENT_ID = 2
YEAR = 2026

WINDOWS = (40, 24, 12)

# ------------------------------------------------------------
# 1) Load field (dg_id list) from Odds_and_Results.xlsx
#    NOTE: adjust sheet_name if your file uses a specific tab name
# ------------------------------------------------------------
odds = pd.read_excel(ODDS_XLSX)

# normalize types (common gotcha if excel stores as float)
odds["year"] = pd.to_numeric(odds.get("year"), errors="coerce")
odds["event_id"] = pd.to_numeric(odds.get("event_id"), errors="coerce")
odds["dg_id"] = pd.to_numeric(odds.get("dg_id"), errors="coerce")

field_ids = (
    odds.loc[(odds["year"] == YEAR) & (odds["event_id"] == EVENT_ID), "dg_id"]
    .dropna()
    .astype(int)
    .unique()
    .tolist()
)

print(f"Field size (unique dg_id): {len(field_ids)}")

# ------------------------------------------------------------
# 2) Load rounds and filter to: in-field + round_date < cutoff
# ------------------------------------------------------------
usecols = [
    "round_date",
    "dg_id",
    "player_name",
    "sg_total",
    "sg_putt",            # you asked for it; if missing, we'll handle below
    "birdies",
    "eagles_or_better",
    "round_score",
]

rounds = pd.read_csv(ROUNDS_PATH, usecols=lambda c: c in set(usecols))

# required columns check
required = {"round_date","dg_id","player_name","sg_total","birdies","eagles_or_better","round_score"}
missing_required = required - set(rounds.columns)
if missing_required:
    raise ValueError(f"combined_rounds file is missing required columns: {sorted(missing_required)}")

# sg_putt is optional (because your "needed columns" list didn’t include it)
has_putt = "sg_putt" in rounds.columns

rounds["round_date"] = pd.to_datetime(rounds["round_date"], errors="coerce")
rounds["dg_id"] = pd.to_numeric(rounds["dg_id"], errors="coerce").astype("Int64")

rounds = rounds.dropna(subset=["round_date","dg_id"]).copy()
rounds["dg_id"] = rounds["dg_id"].astype(int)

rounds = rounds.loc[
    (rounds["dg_id"].isin(field_ids)) &
    (rounds["round_date"] < CUTOFF_DATE)
].copy()

# combined birdies + eagles_or_better per round
rounds["b_eob"] = pd.to_numeric(rounds["birdies"], errors="coerce").fillna(0) + \
                  pd.to_numeric(rounds["eagles_or_better"], errors="coerce").fillna(0)

# Sort so "last N" = most recent N before cutoff
rounds = rounds.sort_values(["dg_id", "round_date"], ascending=[True, False])

print(f"Filtered rounds rows: {len(rounds):,}")

# ------------------------------------------------------------
# 3) Compute last-N window means for each player
# ------------------------------------------------------------
def window_means_for_player(g: pd.DataFrame, windows=(40,24,12)) -> pd.Series:
    out = {}

    # stabilize player_name: take most recent non-null
    pn = g["player_name"].dropna()
    out["player_name"] = pn.iloc[0] if len(pn) else None

    for w in windows:
        sub = g.head(w)
        out[f"n_rounds_L{w}"] = len(sub)

        out[f"sg_total_L{w}"] = pd.to_numeric(sub["sg_total"], errors="coerce").mean()
        out[f"b_eob_L{w}"]    = pd.to_numeric(sub["b_eob"], errors="coerce").mean()
        out[f"round_score_L{w}"] = pd.to_numeric(sub["round_score"], errors="coerce").mean()

        if has_putt:
            out[f"sg_putt_L{w}"] = pd.to_numeric(sub["sg_putt"], errors="coerce").mean()
        else:
            out[f"sg_putt_L{w}"] = np.nan

    return pd.Series(out)

summary = (
    rounds.groupby("dg_id", sort=False, as_index=True)
          .apply(window_means_for_player, windows=WINDOWS)
          .reset_index()
)

# ------------------------------------------------------------
# 4) Ensure every field player appears (even if 0 rounds pre-cutoff)
# ------------------------------------------------------------
field_df = pd.DataFrame({"dg_id": field_ids})
out = field_df.merge(summary, on="dg_id", how="left")

# Optional: order columns cleanly
col_order = ["dg_id", "player_name"]
for w in WINDOWS:
    col_order += [
        f"n_rounds_L{w}",
        f"sg_total_L{w}",
        f"b_eob_L{w}",
        f"sg_putt_L{w}",
        f"round_score_L{w}",
    ]
out = out[col_order]

out.sort_values("sg_total_L40", ascending=False, inplace=True, na_position="last")
out.reset_index(drop=True, inplace=True)

out.head(20)


Field size (unique dg_id): 156
Filtered rounds rows: 37,256


/var/folders/85/gv9dnstn1tn96gql2f7mj15h0000gn/T/ipykernel_23576/2106402628.py:52: DtypeWarning: Columns (36) have mixed types. Specify dtype option on import or set low_memory=False.
  rounds = pd.read_csv(ROUNDS_PATH, usecols=lambda c: c in set(usecols))
/var/folders/85/gv9dnstn1tn96gql2f7mj15h0000gn/T/ipykernel_23576/2106402628.py:110: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(window_means_for_player, windows=WINDOWS)


,dg_id,player_name,n_rounds_L40,sg_total_L40,b_eob_L40,sg_putt_L40,round_score_L40,n_rounds_L24,sg_total_L24,b_eob_L24,sg_putt_L24,round_score_L24,n_rounds_L12,sg_total_L12,b_eob_L12,sg_putt_L12,round_score_L12
0,18417,"Scheffler, Scottie",40,2.994050,5.225000,0.624725,67.525000,24,3.110625,5.875000,0.853542,66.666667,12,2.429083,6.333333,0.572750,66.916667
1,17646,"Fitzpatrick, Matthew",40,1.931775,4.975000,0.793353,68.125000,24,2.178542,5.541667,1.239000,67.958333,12,1.584917,5.333333,NaN,68.166667
2,24968,"Griffin, Ben",40,1.794350,4.600000,0.674406,67.925000,24,1.991583,4.916667,0.720688,67.708333,12,2.053417,5.250000,-0.003000,67.500000
3,10419,"Noren, Alex",40,1.707675,5.175000,1.180944,68.075000,24,1.378583,5.125000,1.171833,68.291667,12,1.417583,5.916667,1.905500,67.833333
4,23838,"Hojgaard, Rasmus",40,1.618600,5.050000,1.147214,68.600000,24,1.547125,5.250000,2.297000,68.250000,12,1.290833,5.083333,NaN,68.333333
5,14578,"Henley, Russell",40,1.399050,3.800000,0.135575,69.250000,24,1.409417,4.166667,0.229667,68.416667,12,1.443333,4.833333,0.248833,68.000000
6,23323,"MacIntyre, Robert",40,1.299625,5.125000,0.838385,68.175000,24,1.236500,5.500000,0.337900,68.333333,12,1.132583,5.500000,0.679125,68.250000
7,27194,"Hall, Harry",40,1.264125,4.625000,1.136419,68.600000,24,0.833542,4.333333,0.725067,68.916667,12,0.831583,3.916667,0.687250,69.416667
8,26649,"Thorbjornsen, Michael",40,1.261650,4.725000,0.044548,68.425000,24,1.527167,4.750000,-0.246733,68.083333,12,1.190833,4.583333,0.242571,67.583333
9,23950,"Aberg, Ludvig",40,1.252550,4.775000,0.289750,68.575000,24,1.222875,5.041667,0.388583,68.208333,12,1.406750,5.583333,NaN,68.333333


In [2]:
import pandas as pd
xls = pd.ExcelFile(ODDS_XLSX)
xls.sheet_names


['Sheet1']

In [3]:
# Excel
OUT_XLSX = Path("/Data/Archive/ae_2026_field_last40_24_12_pre_2026-01-22.xlsx")
out.to_excel(OUT_XLSX, index=False)
print("Saved:", OUT_XLSX)


Saved: /Users/joshmacbook/python_projects/OAD/Data/in Use/ae_2026_field_last40_24_12_pre_2026-01-22.xlsx


In [1]:
from Scripts.data_io import load_rounds

df = load_rounds()

print("shape:", df.shape)
print("\ncolumns:")
print(df.columns.tolist())

# sanity: show dtypes for the ones we care about
cols = [
    "tour","year","round_date","event_completed","event_id",
    "dg_id","player_name",
    "round_score",
    "sg_total","sg_t2g","sg_ott","sg_app","sg_arg","sg_putt",
    "birdies","eagles_or_better",
]
present = [c for c in cols if c in df.columns]
missing = [c for c in cols if c not in df.columns]
print("\npresent:", present)
print("missing:", missing)

print("\ndtypes (present only):")
print(df[present].dtypes)


shape: (311660, 37)

columns:
['tour', 'year', 'season', 'event_completed', 'event_name', 'event_id', 'player_name', 'dg_id', 'fin_text', 'round_num', 'course_name', 'course_num', 'course_par', 'start_hole', 'teetime', 'round_score', 'sg_putt', 'sg_arg', 'sg_app', 'sg_ott', 'sg_t2g', 'sg_total', 'driving_dist', 'driving_acc', 'gir', 'scrambling', 'prox_rgh', 'prox_fw', 'great_shots', 'poor_shots', 'eagles_or_better', 'birdies', 'pars', 'bogies', 'doubles_or_worse', 'finish_num', 'round_date']

present: ['tour', 'year', 'round_date', 'event_completed', 'event_id', 'dg_id', 'player_name', 'round_score', 'sg_total', 'sg_t2g', 'sg_ott', 'sg_app', 'sg_arg', 'sg_putt', 'birdies', 'eagles_or_better']
missing: []

dtypes (present only):
tour                        object
year                         int64
round_date          datetime64[ns]
event_completed     datetime64[ns]
event_id                     int64
dg_id                        int64
player_name                 object
round_score     

In [1]:
import pandas as pd
from pathlib import Path

# adjust if your raw file is named differently
RAW = Path("/Users/joshmacbook/python_projects/OAD/Data/Clean/Leagues/2026_small.csv")
OUT = Path("/Users/joshmacbook/python_projects/OAD/Data/Clean/Leagues/2026_small_normalized.csv")

from Scripts.OAD import load_schedule, normalize_league_df  # where it's defined/used

season = 2026
sched = load_schedule(season)

df_raw = pd.read_csv(RAW, low_memory=False)

df_norm = normalize_league_df(
    df_raw,
    sched=sched,
    season=season,
    league_id_value="main",
)

OUT.parent.mkdir(parents=True, exist_ok=True)
df_norm.to_csv(OUT, index=False)

print("Wrote:", OUT)
print("Columns:", df_norm.columns.tolist())
print("Missing event_id rows:", df_norm["event_id"].isna().sum() if "event_id" in df_norm.columns else "no event_id col")



2026-01-25 22:24:18.089 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-01-25 22:24:18.090 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-01-25 22:24:18.091 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-01-25 22:24:18.091 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-01-25 22:24:18.092 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-01-25 22:24:18.092 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-01-25 22:24:18.093 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-01-25 22:24:18.093 WARNING streamlit.runtime.caching.cache_d

COURSE_SENSITIVITY_PATH: /Users/joshmacbook/python_projects/OAD/Data/Clean/Processed/course_sensitivity_table.csv
  exists?: True


2026-01-25 22:24:18.289 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-01-25 22:24:18.289 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-01-25 22:24:18.289 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-01-25 22:24:18.289 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-01-25 22:24:18.290 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-01-25 22:24:18.290 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-01-25 22:24:18.290 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
/Users/joshmacbook/python_projects/OAD/Scripts/features.py:101: FutureWarning: DataFrameGroupBy.apply operated on the 

NameError: name 'latest_snapshot_upto' is not defined